# Filtrelenmiş BPE v10 Oluşturma Notebook'u

Bu notebook şunları yapar:
1. `new_custom_bpe_tokenizer_yusuf.json` dosyasından vocab çıkarır
2. Test veri setlerinde token frekanslarını hesaplar
3. **İstenmeyen tokenları filtreler:**
   - Latin alfabesi dışındaki karakterleri çıkarır (Kiril, İbranice vb.)
   - Tekrarlanan noktalama işaretlerini temizler (!! -> !)
   - Çoklu rakam kombinasyonlarını filtreler (00, 01, 02 vb.)
   - kokler_v08.json'da bulunan tokenları çıkarır
4. Frekans analizine dayalı olarak temizlenmiş tokenleri seçer
5. `bpe_v10.json` dosyasını oluşturur (ID aralığı: 22869-32767)


In [1]:
# Gerekli kütüphaneleri import et
import json
import os
import re
import string
from collections import Counter, defaultdict
from datasets import load_dataset
from tokenizers import Tokenizer
import pandas as pd
from tqdm import tqdm
import numpy as np


## 1. Mevcut Tokenizer ve Kokler Sözlüğünü Yükle


In [2]:
# new_custom_bpe_tokenizer_yusuf.json dosyasını yükle
print("🔄 Tokenizer dosyası yükleniyor...")
with open('new_custom_bpe_tokenizer_yusuf.json', 'r', encoding='utf-8') as f:
    tokenizer_data = json.load(f)

# Vocab kısmını çıkar
existing_vocab = tokenizer_data['model']['vocab']
print(f"📊 Mevcut vocab boyutu: {len(existing_vocab):,}")

# Tokenizer objesini oluştur
tokenizer = Tokenizer.from_file('new_custom_bpe_tokenizer_yusuf.json')
print("✅ Tokenizer başarıyla yüklendi")

# kokler_v08.json dosyasını yükle
print("\n📚 kokler_v08.json dosyası yükleniyor...")
with open('kokler_v08.json', 'r', encoding='utf-8') as f:
    kokler_dict = json.load(f)

# Kokler sözlüğündeki tokenleri set olarak kaydet (hızlı arama için)
existing_tokens = set(kokler_dict.keys())
print(f"✅ Kokler sözlüğü yüklendi: {len(existing_tokens):,} token")


🔄 Tokenizer dosyası yükleniyor...
📊 Mevcut vocab boyutu: 10,001
✅ Tokenizer başarıyla yüklendi

📚 kokler_v08.json dosyası yükleniyor...
✅ Kokler sözlüğü yüklendi: 24,480 token


## 2. Token Filtreleme Fonksiyonlarını Tanımla


In [3]:
def is_valid_token(token):
    """Token'ın geçerli olup olmadığını kontrol eder"""
    
    # Boş string kontrolü
    if not token or len(token.strip()) == 0:
        return False
    
    # kokler_v08.json'da bulunan tokenları çıkar
    if token in existing_tokens:
        return False
    
    # Latin alfabesi + Türkçe karakterler + rakamlar + noktalama + boşluk
    turkish_chars = 'ÇĞIİÖŞÜçğıiöşü'
    valid_chars = string.ascii_letters + turkish_chars + string.digits + string.punctuation + string.whitespace
    
    # Token'daki her karakteri kontrol et
    for char in token:
        if char not in valid_chars:
            return False
    
    # Tekrarlanan noktalama işaretlerini filtrele
    if len(token) > 1 and all(c in string.punctuation for c in token):
        # Tüm karakteri aynı noktalama işareti ise (örn: "!!", "...", "---")
        if len(set(token)) == 1:
            return False
        # Sadece noktalama işaretlerinden oluşan çok karakterli tokenlar
        return False
    
    # Çoklu rakam kombinasyonlarını filtrele (00, 000, 01, 02, vb.)
    # Sadece tek rakamları kabul et
    if len(token) > 1 and token.isdigit():
        return False
    
    # Sayı benzeri tokenleri filtrele (1., 2., 3. gibi)
    if len(token) > 1 and token[:-1].isdigit() and token[-1] == '.':
        return False
    
    # Kombinasyon noktalama + rakamları filtrele
    if len(token) > 1:
        has_digit = any(c.isdigit() for c in token)
        has_punct = any(c in string.punctuation for c in token)
        if has_digit and has_punct:
            return False
    
    return True

def clean_token_frequencies(token_frequencies):
    """Token frekanslarını filtreler ve temizler"""
    print("🧹 Token frekansları filtreleniyor...")
    
    original_count = len(token_frequencies)
    filtered_frequencies = {}
    
    removed_categories = {
        'non_latin': 0,
        'existing_in_kokler': 0, 
        'repeated_punctuation': 0,
        'multi_digit': 0,
        'other': 0
    }
    
    turkish_chars = 'ÇĞIİÖŞÜçğıiöşü'
    valid_chars = string.ascii_letters + turkish_chars + string.digits + string.punctuation + string.whitespace
    
    for token, freq in token_frequencies.items():
        if not is_valid_token(token):
            # Detaylı kategorilere ayır
            if token in existing_tokens:
                removed_categories['existing_in_kokler'] += 1
            elif not all(c in valid_chars for c in token):
                removed_categories['non_latin'] += 1
            elif len(token) > 1 and token.isdigit():
                removed_categories['multi_digit'] += 1 
            elif len(token) > 1 and all(c in string.punctuation for c in token):
                removed_categories['repeated_punctuation'] += 1
            else:
                removed_categories['other'] += 1
        else:
            filtered_frequencies[token] = freq
    
    filtered_count = len(filtered_frequencies)
    removed_count = original_count - filtered_count
    
    print(f"📊 Filtreleme sonuçları:")
    print(f"   Orijinal token sayısı: {original_count:,}")
    print(f"   Filtrelenmiş token sayısı: {filtered_count:,}")
    print(f"   Çıkarılan token sayısı: {removed_count:,}")
    print(f"   📋 Çıkarılan tokenların kategorileri:")
    print(f"      - kokler_v08.json'da mevcut: {removed_categories['existing_in_kokler']:,}")
    print(f"      - Latin alfabesi dışı karakterler: {removed_categories['non_latin']:,}")
    print(f"      - Tekrarlanan noktalama: {removed_categories['repeated_punctuation']:,}")
    print(f"      - Çoklu rakam kombinasyonları: {removed_categories['multi_digit']:,}")
    print(f"      - Diğer: {removed_categories['other']:,}")
    
    return filtered_frequencies

print("✅ Filtreleme fonksiyonları hazırlandı")


✅ Filtreleme fonksiyonları hazırlandı


## 3. Test Veri Setlerini Yükle ve Token Frekanslarını Hesapla


In [4]:
def sample_and_tokenize(df, text_column, max_samples=50000, dataset_name=""):
    """DataFrame'den örneklem alıp tokenize eder"""
    print(f"🔄 {dataset_name} tokenize ediliyor...")
    
    # Örneklem al
    sample_size = min(len(df), max_samples)
    df_sample = df.head(sample_size)
    
    all_tokens = []
    
    for i in tqdm(range(len(df_sample)), desc=f"{dataset_name} işleniyor"):
        try:
            text = df_sample.iloc[i][text_column]
            if isinstance(text, str) and len(text.strip()) > 0:
                
                # Tokenize et
                encoded = tokenizer.encode(text)
                tokens = encoded.tokens
                all_tokens.extend(tokens)
                
        except Exception as e:
            continue
    
    print(f"✅ {dataset_name}: {len(all_tokens):,} token")
    return all_tokens

# Frekans analizi için büyük veri setlerini yükle
print("📚 Test veri setleri yükleniyor...")

# Wikipedia veri seti
print("\n🔄 Wikipedia yükleniyor...")
dswiki = load_dataset("wikimedia/wikipedia", "20231101.tr")
dfwiki = dswiki['train'].to_pandas()
print(f"✅ Wikipedia: {len(dfwiki):,} makale")

# Yorumlar veri seti (frekans analizi için)
print("\n🔄 Hepsiburada yorumları yükleniyor...")
dshepsi = load_dataset("alibayram/hepsiburada_yorumlar")
dfhepsi = dshepsi['train'].to_pandas()
print(f"✅ Hepsiburada: {len(dfhepsi):,} yorum")

print("\n🔄 Beyazperde yorumları yükleniyor...")
dsbeyazperde = load_dataset("alibayram/beyazperde_yorumlar")
dfbeyazperde = dsbeyazperde['train'].to_pandas()
print(f"✅ Beyazperde: {len(dfbeyazperde):,} yorum")

print("\n🔄 Kitapyurdu yorumları yükleniyor...")
dskitapyurdu = load_dataset("alibayram/kitapyurdu_yorumlar")
dfkitapyurdu = dskitapyurdu['train'].to_pandas()
print(f"✅ Kitapyurdu: {len(dfkitapyurdu):,} yorum")

print("\n🔄 Yorumbudur yorumları yükleniyor...")
dsyorumbudur= load_dataset("alibayram/yorumbudur")
dfyorumbudur = dsyorumbudur['train'].to_pandas()
print(f"✅ Yorumbudur: {len(dfyorumbudur):,} yorum")

print(f"\n📊 Toplam veri boyutu: {len(dfwiki) + len(dfhepsi):,} metin")


📚 Test veri setleri yükleniyor...

🔄 Wikipedia yükleniyor...
✅ Wikipedia: 534,988 makale

🔄 Hepsiburada yorumları yükleniyor...
✅ Hepsiburada: 2,657,073 yorum

🔄 Beyazperde yorumları yükleniyor...
✅ Beyazperde: 192,074 yorum

🔄 Kitapyurdu yorumları yükleniyor...
✅ Kitapyurdu: 404,637 yorum

🔄 Yorumbudur yorumları yükleniyor...
✅ Yorumbudur: 2,563,449 yorum

📊 Toplam veri boyutu: 3,192,061 metin


In [5]:
# Token frekanslarını hesapla
print("📊 Token frekansları hesaplanıyor...\n")
token_frekanslari = Counter()

# Wikipedia tokenları
wiki_tokens = sample_and_tokenize(dfwiki, 'text', max_samples=5000000, dataset_name="Wikipedia")
token_frekanslari.update(wiki_tokens)

# Yorum tokenları
yorum_tokens = sample_and_tokenize(dfhepsi, 'Yorum', max_samples=20000000, dataset_name="Yorumlar")
token_frekanslari.update(yorum_tokens)

# Beyazperde tokenları
beyazperde_tokens = sample_and_tokenize(dfbeyazperde, 'Yorum', max_samples=10000000, dataset_name="Beyazperde")
token_frekanslari.update(beyazperde_tokens)

# Kitapyurdu tokenları
kitapyurdu_tokens = sample_and_tokenize(dfkitapyurdu, 'Yorum', max_samples=40000000, dataset_name="Kitapyurdu")
token_frekanslari.update(kitapyurdu_tokens)

# Yorumbudur tokenları
yorumbudur_tokens = sample_and_tokenize(dfyorumbudur, 'Yorum', max_samples=20000000, dataset_name="Yorumbudur")
token_frekanslari.update(yorumbudur_tokens)

print(f"\n📊 Frekans analizi tamamlandı:")
print(f"   Toplam farklı token: {len(token_frekanslari):,}")
print(f"   Toplam token sayısı: {sum(token_frekanslari.values()):,}")


📊 Token frekansları hesaplanıyor...

🔄 Wikipedia tokenize ediliyor...


Wikipedia işleniyor: 100%|██████████| 534988/534988 [03:42<00:00, 2407.84it/s] 


✅ Wikipedia: 233,537,369 token
🔄 Yorumlar tokenize ediliyor...


Yorumlar işleniyor: 100%|██████████| 2657073/2657073 [02:01<00:00, 21910.06it/s]


✅ Yorumlar: 77,803,788 token
🔄 Beyazperde tokenize ediliyor...


...

✅ Beyazperde: 11,547,569 token
🔄 Kitapyurdu tokenize ediliyor...


Kitapyurdu işleniyor: 100%|██████████| 404637/404637 [00:14<00:00, 27741.15it/s]


✅ Kitapyurdu: 9,079,026 token
🔄 Yorumbudur tokenize ediliyor...


Yorumbudur işleniyor: 100%|██████████| 2563449/2563449 [02:00<00:00, 21207.54it/s]

✅ Yorumbudur: 82,198,207 token

📊 Frekans analizi tamamlandı:
   Toplam farklı token: 9,890
   Toplam token sayısı: 414,165,959


## 4. Token Frekanslarını Filtrele ve En Sık Kullanılanları Belirle


In [11]:
# Token frekanslarını filtrele
filtered_token_frequencies = clean_token_frequencies(token_frekanslari)

# Filtrelenmiş tokenleri frekansa göre sırala
sirali_tokenlar = sorted(filtered_token_frequencies.items(), key=lambda x: x[1], reverse=True)

print("\n🏆 Filtreleme sonrası en sık kullanılan 20 token:")
for i, (token, freq) in enumerate(sirali_tokenlar[:20]):
    print(f"{i+1:2d}. '{token}' - {freq:,} kez")

# Frekans dağılımını analiz et
freqs = list(filtered_token_frequencies.values())
print(f"\n📈 Filtrelenmiş frekans istatistikleri:")
print(f"   En yüksek: {max(freqs):,}")
print(f"   En düşük: {min(freqs):,}")
print(f"   Ortalama: {np.mean(freqs):.1f}")
print(f"   Medyan: {np.median(freqs):.1f}")

# Frekans eşiği belirle (dinamik)
# BPE v10 için 9899 token hedefliyoruz (22869-32767)
BPE_START_ID = 22869
BPE_END_ID = 32767
TARGET_TOKENS = BPE_END_ID - BPE_START_ID + 1

print(f"\n🎯 Hedef token sayısı: {TARGET_TOKENS:,}")

# En sık kullanılan TARGET_TOKENS kadar token al
if len(sirali_tokenlar) > TARGET_TOKENS:
    secili_tokenlar = sirali_tokenlar[:TARGET_TOKENS]
    min_frekans = secili_tokenlar[-1][1]
else:
    secili_tokenlar = sirali_tokenlar
    min_frekans = 1

print(f"📊 Filtrelenmiş token sayısı: {len(sirali_tokenlar):,}")
print(f"📊 Seçilen token sayısı: {len(secili_tokenlar):,}")
print(f"📊 Minimum frekans eşiği: {min_frekans:,}")

if secili_tokenlar:
    frekanslar = [freq for _, freq in secili_tokenlar]
    print(f"\n📊 Seçilen tokenların istatistikleri:")
    print(f"   En yüksek frekans: {max(frekanslar):,}")
    print(f"   En düşük frekans: {min(frekanslar):,}")
    print(f"   Ortalama frekans: {np.mean(frekanslar):.1f}")

print(f"\n✅ Token seçimi tamamlandı!")


🧹 Token frekansları filtreleniyor...
📊 Filtreleme sonuçları:
   Orijinal token sayısı: 9,890
   Filtrelenmiş token sayısı: 5,290
   Çıkarılan token sayısı: 4,600
   📋 Çıkarılan tokenların kategorileri:
      - kokler_v08.json'da mevcut: 3,207
      - Latin alfabesi dışı karakterler: 1,174
      - Tekrarlanan noktalama: 39
      - Çoklu rakam kombinasyonları: 180
      - Diğer: 0

🏆 Filtreleme sonrası en sık kullanılan 20 token:
 1. '.' - 11,879,527 kez
 2. ',' - 6,734,976 kez
 3. ''' - 4,595,032 kez
 4. 'da' - 2,632,266 kez
 5. 'e' - 2,525,617 kez
 6. 'de' - 2,524,144 kez
 7. 'a' - 2,219,335 kez
 8. 'en' - 2,127,016 kez
 9. 'u' - 2,111,704 kez
10. 'r' - 2,089,853 kez
11. '-' - 2,066,416 kez
12. 'i' - 2,040,951 kez
13. '(' - 1,951,327 kez
14. 'n' - 1,811,542 kez
15. 's' - 1,740,278 kez
16. '"' - 1,734,661 kez
17. 'an' - 1,716,336 kez
18. 'in' - 1,636,241 kez
19. 'ar' - 1,622,817 kez
20. 'k' - 1,526,692 kez

📈 Filtrelenmiş frekans istatistikleri:
   En yüksek: 11,879,527
   En düşük: 1
 

## 5. Filtrelenmiş BPE v10 JSON Dosyasını Oluştur


In [12]:
# BPE v10 dictionary oluştur
print("📝 Filtrelenmiş BPE v10 dictionary oluşturuluyor...")

bpe_v10_dict = {}
current_id = BPE_START_ID

# Seçilen tokenleri ID'leriyle birlikte sözlüğe ekle
for i, (token, freq) in enumerate(secili_tokenlar):
    if current_id <= BPE_END_ID:
        bpe_v10_dict[token] = current_id
        current_id += 1
    else:
        break

print(f"✅ Filtrelenmiş BPE v10 dictionary oluşturuldu")
print(f"📊 Token sayısı: {len(bpe_v10_dict):,}")
print(f"🔢 Kullanılan ID aralığı: {BPE_START_ID} - {current_id - 1}")

# JSON dosyasını kaydet
output_file = 'bpe_v10.json'
print(f"\n💾 {output_file} dosyasına kaydediliyor...")

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(bpe_v10_dict, f, ensure_ascii=False, indent=4, sort_keys=True)

print(f"✅ {output_file} başarıyla oluşturuldu!")
print(f"📊 Dosya boyutu: {os.path.getsize(output_file) / 1024:.1f} KB")

# İlk 10 tokenı göster
print(f"\n📝 İlk 10 Filtrelenmiş BPE v10 tokeni:")
first_10 = list(bpe_v10_dict.items())[:10]
for token, token_id in first_10:
    print(f"   '{token}': {token_id}")
print("   ...")


📝 Filtrelenmiş BPE v10 dictionary oluşturuluyor...
✅ Filtrelenmiş BPE v10 dictionary oluşturuldu
📊 Token sayısı: 5,290
🔢 Kullanılan ID aralığı: 22869 - 28158

💾 bpe_v10.json dosyasına kaydediliyor...
✅ bpe_v10.json başarıyla oluşturuldu!
📊 Dosya boyutu: 111.5 KB

📝 İlk 10 Filtrelenmiş BPE v10 tokeni:
   '.': 22869
   ',': 22870
   ''': 22871
   'da': 22872
   'e': 22873
   'de': 22874
   'a': 22875
   'en': 22876
   'u': 22877
   'r': 22878
   ...


## 6. Sonuç Analizi ve Kalite Kontrolü


In [8]:
# İstatistikleri hesapla
token_lengths = [len(token) for token in bpe_v10_dict.keys()]
avg_token_length = np.mean(token_lengths)
max_token_length = max(token_lengths)
min_token_length = min(token_lengths)

print("📈 Filtrelenmiş BPE v10 İstatistikleri:")
print(f"   Toplam token sayısı: {len(bpe_v10_dict):,}")
print(f"   ID aralığı: {BPE_START_ID} - {max(bpe_v10_dict.values())}")
print(f"   Ortalama token uzunluğu: {avg_token_length:.2f} karakter")
print(f"   En uzun token: {max_token_length} karakter")
print(f"   En kısa token: {min_token_length} karakter")

# En uzun ve en kısa tokenleri göster
longest_tokens = [token for token in bpe_v10_dict.keys() if len(token) == max_token_length][:5]
shortest_tokens = [token for token in bpe_v10_dict.keys() if len(token) == min_token_length][:5]

print(f"\n📏 En uzun tokenler ({max_token_length} karakter):")
for token in longest_tokens:
    print(f"   '{token}'")

print(f"\n📏 En kısa tokenler ({min_token_length} karakter):")
for token in shortest_tokens:
    print(f"   '{token}'")

# En sık kullanılan tokenları göster (frekans bilgisiyle)
print(f"\n🏆 En sık kullanılan 15 token (frekansla):")
for i, (token, freq) in enumerate(secili_tokenlar[:15]):
    token_id = bpe_v10_dict[token]
    print(f"{i+1:2d}. ID {token_id}: '{token}' (frekans: {freq:,})")

print(f"\n🎉 Filtrelenmiş BPE v10 oluşturma işlemi tamamlandı!")
print(f"📁 Dosya konumu: {os.path.abspath(output_file)}")
print(f"📊 Frekans analizine dayalı {len(bpe_v10_dict):,} temizlenmiş token içeriyor")
print(f"🚫 Çıkarılan problemli tokenlar: Latin dışı karakterler, tekrarlanan noktalama, çoklu rakamlar, kokler_v08 tokenleri")


📈 Filtrelenmiş BPE v10 İstatistikleri:
   Toplam token sayısı: 5,290
   ID aralığı: 22869 - 28158
   Ortalama token uzunluğu: 5.87 karakter
   En uzun token: 18 karakter
   En kısa token: 1 karakter

📏 En uzun tokenler (18 karakter):
   'gerçekleştirilecek'
   'değerlendirmesinde'

📏 En kısa tokenler (1 karakter):
   '.'
   ','
   '''
   'e'
   'a'

🏆 En sık kullanılan 15 token (frekansla):
 1. ID 22869: '.' (frekans: 11,879,527)
 2. ID 22870: ',' (frekans: 6,734,976)
 3. ID 22871: ''' (frekans: 4,595,032)
 4. ID 22872: 'da' (frekans: 2,632,266)
 5. ID 22873: 'e' (frekans: 2,525,617)
 6. ID 22874: 'de' (frekans: 2,524,144)
 7. ID 22875: 'a' (frekans: 2,219,335)
 8. ID 22876: 'en' (frekans: 2,127,016)
 9. ID 22877: 'u' (frekans: 2,111,704)
10. ID 22878: 'r' (frekans: 2,089,853)
11. ID 22879: '-' (frekans: 2,066,416)
12. ID 22880: 'i' (frekans: 2,040,951)
13. ID 22881: '(' (frekans: 1,951,327)
14. ID 22882: 'n' (frekans: 1,811,542)
15. ID 22883: 's' (frekans: 1,740,278)

🎉 Filtrelenmiş B

## 7. Test Tokenization


In [9]:
# Test tokenizasyon
test_texts = [
    "Bu bir test metnidir.",
    "Merhaba dünya! Nasılsın?",
    "Türkiye'de yaşıyorum.",
    "Çok güzel bir gün bugün."
]

print("🧪 Test Tokenization:")
for text in test_texts:
    encoded = tokenizer.encode(text)
    tokens = encoded.tokens
    print(f"\n📝 Metin: '{text}'")
    print(f"🔤 Tokenler: {tokens}")
    print(f"🔢 Token sayısı: {len(tokens)}")
    
    # BPE v10'da hangi tokenler var?
    bpe_v10_tokens = [token for token in tokens if token in bpe_v10_dict]
    print(f"✨ Filtrelenmiş BPE v10'da bulunan: {len(bpe_v10_tokens)}/{len(tokens)} token")

print("\n✨ Notebook tamamlandı!")
print("📊 Frekans analizine dayalı, filtrelenmiş BPE v10 dosyası başarıyla oluşturuldu.")
print("🎯 Artık Latin alfabesi dışı karakterler, tekrarlanan noktalama, çoklu rakamlar ve kokler tokenleri temizlendi!")


🧪 Test Tokenization:

📝 Metin: 'Bu bir test metnidir.'
🔤 Tokenler: ['u', 'bir', 'test', 'met', 'ni', 'dir', '.']
🔢 Token sayısı: 7
✨ Filtrelenmiş BPE v10'da bulunan: 4/7 token

📝 Metin: 'Merhaba dünya! Nasılsın?'
🔤 Tokenler: ['er', 'ha', 'ba', 'dünya', '!', 'asıl', 'sın', '?']
🔢 Token sayısı: 8
✨ Filtrelenmiş BPE v10'da bulunan: 4/8 token

📝 Metin: 'Türkiye'de yaşıyorum.'
🔤 Tokenler: ['ür', 'k', 'iye', "'", 'de', 'yaş', 'ıyorum', '.']
🔢 Token sayısı: 8
✨ Filtrelenmiş BPE v10'da bulunan: 6/8 token

📝 Metin: 'Çok güzel bir gün bugün.'
🔤 Tokenler: ['ok', 'güzel', 'bir', 'gün', 'bugün', '.']
🔢 Token sayısı: 6
✨ Filtrelenmiş BPE v10'da bulunan: 1/6 token

✨ Notebook tamamlandı!
📊 Frekans analizine dayalı, filtrelenmiş BPE v10 dosyası başarıyla oluşturuldu.
🎯 Artık Latin alfabesi dışı karakterler, tekrarlanan noktalama, çoklu rakamlar ve kokler tokenleri temizlendi!
